In [37]:
import pandas as pd 
import re
from collections import Counter
from pathlib import Path

In [38]:
df = pd.read_csv("/Users/tam_pr/tcg_bot/data/raw/EN_Card_Data_R2_full.csv")

In [39]:
columns = df.columns
columns

Index(['Card ID', 'Card Name', 'Expansion', 'Collection No.',
       'Stage (Pokémon)/Type (Energy and Trainer)', 'Rule', 'Category',
       'Previous stage', 'HP', 'Type', 'Weakness', 'Resistance (Type)',
       'Retreat', 'Move Name', 'Cost', 'Damage', 'Effect Explanation',
       'Species'],
      dtype='str')

In [40]:
df.shape

(2273, 18)

In [41]:
rename_map = {
    "Card ID": "card_id",
    "Card Name": "card_name",
    "Expansion": "expansion",
    "Collection No.": "collection_no",
    "Stage (Pokémon)/Type (Energy and Trainer)": "card_kind",
    "Rule": "rule",
    "Category": "category",
    "Previous stage": "previous_stage",
    "HP": "hp",
    "Type": "type",
    "Weakness": "weakness",
    "Resistance (Type)": "resistance",
    "Retreat": "retreat_cost",
    "Move Name": "move_name",
    "Cost": "cost",
    "Damage": "damage",
    "Effect Explanation": "effect_text",
    "Species": "species",
}

df = df.rename(columns=rename_map)

In [42]:
print(df.columns.tolist())
assert len(df.columns) == 18
assert df.columns.is_unique
assert not any(c != c.lower() or " " in c for c in df.columns)


['card_id', 'card_name', 'expansion', 'collection_no', 'card_kind', 'rule', 'category', 'previous_stage', 'hp', 'type', 'weakness', 'resistance', 'retreat_cost', 'move_name', 'cost', 'damage', 'effect_text', 'species']


In [43]:
df.shape

(2273, 18)

### Data Sectioning

In Pokémon TCG there is three kinds of primary cards: <br>

1. Trainer cards: There's various kinds: Items, Supporters, Stadiums and Pokémon Tools. 
2. Pokémon cards: There's three kinds: Basic, Stage 1 and Stage 2. 
3. Energy cards: Powers up Pokémons, there's 11 different kinds. 

The dataset we are working on contains all the different kinds of cards, we need to section them in this three main groups so we can process each one with the most optimal strategy. 

<br>

The following sub-datasets will be obtained in this first cut, to divide the kinds of cards 

| # | Table           | Row =               | Key       |
|---|-----------------|---------------------|-----------|
| 1 | `cards`         | one card (any kind) | `card_id` |
| 2 | `pokemon_cards` | one Pokémon card    | `card_id` |
| 3 | `trainer_cards` | one Trainer card    | `card_id` |


In [44]:
print(df["card_kind"].value_counts(dropna=False))

card_kind
Basic Pokémon      1073
Stage 1 Pokémon     699
Stage 2 Pokémon     258
Item                 90
Supporter            69
Pokémon Tool         30
Stadium              29
Special Energy       17
Basic Energy          8
Name: count, dtype: int64


In [45]:
pokemon_cards = ["Basic Pokémon", "Stage 1 Pokémon", "Stage 2 Pokémon"]
trainer_cards = ["Item", "Supporter", "Pokémon Tool", "Stadium"]
energy_cards = ["Basic Energy", "Special Energy"]

In [46]:
pokemon = df[df["card_kind"].isin(pokemon_cards)]
trainer = df[df["card_kind"].isin(trainer_cards)]
energy  = df[df["card_kind"].isin(energy_cards)]

In [47]:
print(len(pokemon), len(trainer), len(energy))
assert len(pokemon) + len(trainer) + len(energy) == len(df)

2030 218 25


In [48]:
assert not (set(pokemon.index) & set(trainer.index))
assert not (set(pokemon.index) & set(energy.index))
assert not (set(trainer.index) & set(energy.index))

In [49]:
leftover = df[~df.index.isin(pokemon.index.union(trainer.index).union(energy.index))]
print(len(leftover)) 

0


In [50]:
for name, part in [("pokemon", pokemon), ("trainer", trainer), ("energy", energy)]:
    print(f"\n{name}")
    print(part.replace("n/a", pd.NA).notna().sum())


pokemon
card_id           2030
card_name         2030
expansion         2021
collection_no     2030
card_kind         2030
rule               355
category           376
previous_stage     957
hp                2030
type              2030
weakness          1953
resistance         443
retreat_cost      1969
move_name         2030
cost              1754
damage            1446
effect_text       1436
species            206
dtype: int64

trainer
card_id           218
card_name         218
expansion         217
collection_no     218
card_kind         218
rule               26
category           20
previous_stage     14
hp                 14
type                0
weakness            0
resistance          0
retreat_cost        0
move_name           8
cost                1
damage              1
effect_text       218
species             0
dtype: int64

energy
card_id           25
card_name         25
expansion         25
collection_no     25
card_kind         25
rule               3
category    

Now the data set is sectioned into three subdata sets, divided by the card type and its atributes 

In [51]:
pokemon.head(10)

,card_id,card_name,expansion,collection_no,card_kind,rule,category,previous_stage,hp,type,weakness,resistance,retreat_cost,move_name,cost,damage,effect_text,species
20,21,Scrafty,PROMO,188,Stage 1 Pokémon,NaN,NaN,Scraggy,120.0,{D},{G},NaN,1.0,Nab ’n’ Dash,●,NaN,Search your deck for a number of cards up to t...,NaN
21,21,Scrafty,PROMO,188,Stage 1 Pokémon,NaN,NaN,Scraggy,120.0,{D},{G},NaN,1.0,High Jump Kick,{D}●●,100,NaN,NaN
22,22,Hippopotas,DRI,105,Basic Pokémon,NaN,NaN,NaN,90.0,{F},{G},NaN,3.0,Push Down,{F},10,Switch out your opponent’s Active Pokémon to t...,NaN
23,23,Hippowdon,DRI,106,Stage 1 Pokémon,NaN,NaN,Hippopotas,160.0,{F},{G},NaN,4.0,Ram,{F}{F},60,NaN,NaN
24,23,Hippowdon,DRI,106,Stage 1 Pokémon,NaN,NaN,Hippopotas,160.0,{F},{G},NaN,4.0,Super Sandstorm,{F}{F}●,150,This attack also does 40 damage to each Benche...,NaN
25,24,Team Rocket's Kangaskhan ex,ASC,162,Basic Pokémon,Pokémon ex,Trainer's Pokémon(Team Rocket),NaN,230.0,{C},{F},NaN,2.0,\r\nComet Punch,●●,30×,Flip 4 coins. This attack does 30 damage for e...,NaN
26,24,Team Rocket's Kangaskhan ex,ASC,162,Basic Pokémon,Pokémon ex,Trainer's Pokémon(Team Rocket),NaN,230.0,{C},{F},NaN,2.0,Wicked Impact,●●●,120,If you played a Supporter card that has “Team ...,NaN
27,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Slow Crunch,{G}●,NaN,Discard all Energy from this Pokémon. At the e...,NaN
28,25,Pinsir,TWM,3,Basic Pokémon,NaN,NaN,NaN,110.0,{G},{R},NaN,2.0,Superpowered Horns,{G}●●,100,NaN,NaN
29,26,Leafeon,TWM,11,Stage 1 Pokémon,NaN,NaN,Eevee,120.0,{G},{R},NaN,1.0,Leaflet Blessings,●,NaN,Attach a Basic {G} Energy card from your hand ...,NaN


String treatment 

In [52]:
text_cols = df.select_dtypes(include=["object", "string"]).columns
for c in text_cols:
    df[c] = (
        df[c].astype("string")
        .str.replace("\u2019", "'", regex=False)   # curly -> straight apostrophe
        .str.replace(r"\s+", " ", regex=True)      # collapse newlines / double spaces
        .str.strip()
        .replace({"n/a": pd.NA, "": pd.NA})
    )

Numeric values treatment 

In [53]:
for c in ["hp", "retreat_cost", "collection_no"]:
    df[c] = pd.to_numeric(df[c], errors="raise").astype("Int64")

# For Pokémon, a blank retreat cost means 0 (e.g. Shaymin, Budew), not missing
pokemon_kinds = ["Basic Pokémon", "Stage 1 Pokémon", "Stage 2 Pokémon"]
is_pokemon = df["card_kind"].isin(pokemon_kinds)
df.loc[is_pokemon, "retreat_cost"] = df.loc[is_pokemon, "retreat_cost"].fillna(0)

Move types 


In [54]:
prefix = df["move_name"].str.extract(r"^\[(\w+)\]")[0].str.lower()
has_move = df["move_name"].notna()

df["move_kind"] = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[has_move, "move_kind"] = "attack"
df.loc[prefix.notna(), "move_kind"] = prefix[prefix.notna()]

df["move_name"] = df["move_name"].str.replace(r"^\[\w+\]\s*", "", regex=True)
df["move_kind"].value_counts(dropna=False)

move_kind
attack     1755
ability     251
<NA>        235
tera         32
Name: count, dtype: Int64

Encoding the energy types requirements 

In [55]:
def parse_cost(s):
    if pd.isna(s) or s == "No cost":
        return {}
    counts = Counter(re.findall(r"\{([A-Za-z ]+)\}", s))
    counts["C"] += s.count("●")          # ● = colorless
    return dict(counts)

cost_counts = pd.DataFrame(df["cost"].map(parse_cost).tolist(), index=df.index)
cost_counts = cost_counts.fillna(0).astype("Int64").add_prefix("cost_")
cost_counts = cost_counts.where(df["cost"].notna())     # keep NA when there is no cost field at all

df = pd.concat([df, cost_counts], axis=1)
df["cost_total"] = cost_counts.sum(axis=1, min_count=1).astype("Int64")
df["no_cost"] = (df["cost"] == "No cost").fillna(False).astype(bool)   # 5 rows are literally "No cost"

df[["cost", *cost_counts.columns, "cost_total"]].drop_duplicates("cost").head(8)

,cost,cost_C,cost_D,cost_F,cost_G,cost_R,cost_W,cost_L,cost_P,cost_M,cost_total
0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
20,●,1,0,0,0,0,0,0,0,0,1
21,{D}●●,2,1,0,0,0,0,0,0,0,3
22,{F},0,0,1,0,0,0,0,0,0,1
23,{F}{F},0,0,2,0,0,0,0,0,0,2
24,{F}{F}●,1,0,2,0,0,0,0,0,0,3
25,●●,2,0,0,0,0,0,0,0,0,2
26,●●●,3,0,0,0,0,0,0,0,0,3


Damage encoding 

In [56]:
dmg = df["damage"].str.extract(r"^(?P<damage_base>\d+)(?P<sym>[×x+\-]?)$")

unparsed = df["damage"].notna() & dmg["damage_base"].isna()
assert not unparsed.any(), df.loc[unparsed, "damage"].unique()

df["damage_base"] = dmg["damage_base"].astype("Int64")
df["damage_mod"] = (
    dmg["sym"].map({"": "flat", "×": "times", "x": "times", "+": "plus", "-": "minus"})
    .astype("string")
)
df["damage_mod"].value_counts(dropna=False)

damage_mod
flat     1323
<NA>      826
times     121
minus       3
Name: count, dtype: Int64

Rules and constraints within the cards

In [57]:
flag = lambda s: s.fillna(False).astype(bool)

df["is_ex"]       = flag(df["rule"].isin(["Pokémon ex", "Mega Pokémon ex"]))
df["is_mega"]     = flag(df["rule"] == "Mega Pokémon ex")
df["is_ace_spec"] = flag(df["rule"] == "ACE SPEC")

df["is_tera"]     = flag(df["category"].str.startswith("Tera"))
df["tera_type"]   = df["category"].str.extract(r"^Tera\((.+)\)$")[0]
df["trainer_owner"] = df["category"].str.extract(r"^Trainer's Pokémon\((.+)\)$")[0]

for k in ["Ancient", "Future", "Fossil"]:
    df[f"is_{k.lower()}"] = flag(df["category"] == k)

df[["is_ex", "is_mega", "is_ace_spec", "is_tera", "is_ancient", "is_future", "is_fossil"]].sum()

is_ex          355
is_mega         70
is_ace_spec     29
is_tera         96
is_ancient      26
is_future       17
is_fossil       14
dtype: int64

Integrates into three data frames

In [58]:
trainer_kinds = ["Item", "Supporter", "Pokémon Tool", "Stadium"]
energy_kinds  = ["Basic Energy", "Special Energy"]

pokemon = df[df["card_kind"].isin(pokemon_kinds)].copy()
trainer = df[df["card_kind"].isin(trainer_kinds)].copy()
energy  = df[df["card_kind"].isin(energy_kinds)].copy()

assert len(pokemon) + len(trainer) + len(energy) == len(df)
assert (pokemon["retreat_cost"].notna()).all()
assert pokemon["hp"].notna().all()
assert pokemon["move_kind"].notna().all()
print(len(pokemon), len(trainer), len(energy))   # expect 2030 218 25

2030 218 25


#### Building the moves and cards data frames

| # | Table   | Row =                          | Key                |
|---|---------|--------------------------------|--------------------|
| 1 | `cards` | one card (any kind)            | `card_id`          |
| 2 | `moves` | one attack, ability or effect  | `move_id`          |

In [59]:
family_map = (
    {k: "pokemon" for k in pokemon_kinds}
    | {k: "trainer" for k in trainer_kinds}
    | {k: "energy" for k in energy_kinds}
)
df["family"] = df["card_kind"].map(family_map)
assert df["family"].notna().all()
df["family"].value_counts()

family
pokemon    2030
trainer     218
energy       25
Name: count, dtype: int64

In [60]:
# a Trainer row that has an attack name (Core Memory's Geobuster) is a *granted* attack
df.loc[(df["family"] != "pokemon") & (df["move_kind"] == "attack"), "move_kind"] = "granted_attack"

# Trainer / Energy rows with effect text but no move name are plain card effects
no_move = df["move_kind"].isna() & df["effect_text"].notna()
df.loc[no_move & (df["family"] == "trainer"), "move_kind"] = "trainer_effect"
df.loc[no_move & (df["family"] == "energy"), "move_kind"] = "energy_effect"

df["move_kind"].value_counts(dropna=False)

move_kind
attack            1754
ability            251
trainer_effect     210
tera                32
energy_effect       17
<NA>                 8
granted_attack       1
Name: count, dtype: Int64

In [61]:
cost_cols = [c for c in df.columns if c.startswith("cost_") and c != "cost_total"]

# rows with neither a move nor an effect (the 8 Basic Energy) are not moves
moves = df[df["move_kind"].notna()].copy()
moves["move_idx"] = moves.groupby("card_id").cumcount()
moves["move_id"] = moves["card_id"].astype(str) + "_" + moves["move_idx"].astype(str)

moves = moves[[
    "move_id", "card_id", "move_idx", "move_kind", "move_name", "effect_text",
    "damage_base", "damage_mod", "no_cost", "cost_total", *cost_cols,
]].reset_index(drop=True)

print(moves.shape)   # expect (2265, 19)
moves.head(30)

(2265, 19)


,move_id,card_id,move_idx,move_kind,move_name,effect_text,damage_base,damage_mod,no_cost,cost_total,cost_C,cost_D,cost_F,cost_G,cost_R,cost_W,cost_L,cost_P,cost_M
0,9_0,9,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,10_0,10,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,11_0,11,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,12_0,12,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,13_0,13,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
5,14_0,14,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
6,15_0,15,0,energy_effect,<NA>,This card can only be attached to a Team Rocke...,<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
7,16_0,16,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
8,17_0,17,0,energy_effect,<NA>,"If this card is attached to 1 of your Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
9,18_0,18,0,energy_effect,<NA>,"As long as this card is attached to a Pokémon,...",<NA>,<NA>,False,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


In [62]:
card_cols = [
    "card_name", "expansion", "collection_no", "family", "card_kind", "rule", "category",
    "previous_stage", "hp", "type", "weakness", "resistance", "retreat_cost", "species",
    "is_ex", "is_mega", "is_ace_spec", "is_tera", "tera_type", "trainer_owner",
    "is_ancient", "is_future", "is_fossil",
]

# card-level columns must agree across a card's rows (blank on one row is fine)
conflicts = df.groupby("card_id")[card_cols].nunique(dropna=True).gt(1).sum()
assert conflicts.sum() == 0, conflicts[conflicts > 0]

cards = df.groupby("card_id", sort=False)[card_cols].first().reset_index()
cards["n_moves"] = cards["card_id"].map(moves.groupby("card_id").size()).fillna(0).astype(int)

print(cards.shape)   # expect (1431, 25)
cards.head()

(1431, 25)


,card_id,card_name,expansion,collection_no,family,card_kind,rule,category,previous_stage,hp,...,is_ex,is_mega,is_ace_spec,is_tera,tera_type,trainer_owner,is_ancient,is_future,is_fossil,n_moves
0,1,Basic {G} Energy,SVE,1,energy,Basic Energy,<NA>,<NA>,<NA>,<NA>,...,False,False,False,False,<NA>,<NA>,False,False,False,0
1,2,Basic {R} Energy,SVE,2,energy,Basic Energy,<NA>,<NA>,<NA>,<NA>,...,False,False,False,False,<NA>,<NA>,False,False,False,0
2,3,Basic {W} Energy,SVE,3,energy,Basic Energy,<NA>,<NA>,<NA>,<NA>,...,False,False,False,False,<NA>,<NA>,False,False,False,0
3,4,Basic {L} Energy,SVE,4,energy,Basic Energy,<NA>,<NA>,<NA>,<NA>,...,False,False,False,False,<NA>,<NA>,False,False,False,0
4,5,Basic {P} Energy,SVE,5,energy,Basic Energy,<NA>,<NA>,<NA>,<NA>,...,False,False,False,False,<NA>,<NA>,False,False,False,0


In [63]:
assert cards["card_id"].is_unique
assert moves["move_id"].is_unique
assert len(cards) == df["card_id"].nunique()
assert moves["card_id"].isin(cards["card_id"]).all()
assert (cards.loc[cards["family"] == "pokemon", "n_moves"] >= 1).all()
assert (cards.loc[cards["family"] == "trainer", "n_moves"] >= 1).all()
for t in (cards, moves):
    for c in t.select_dtypes("string").columns:
        assert not (t[c] == "n/a").any(), c

print(cards["family"].value_counts().to_dict())   # {'pokemon': 1196, 'trainer': 210, 'energy': 25}
print(cards.shape, moves.shape)                   # (1431, 25) (2265, 19)

{'pokemon': 1196, 'trainer': 210, 'energy': 25}
(1431, 25) (2265, 19)


In [64]:
out = Path("/Users/tam_pr/tcg_bot/data/processed")
out.mkdir(parents=True, exist_ok=True)

cards.to_parquet(out / "cards.parquet", index=False)
moves.to_parquet(out / "moves.parquet", index=False)

# read back and compare
assert pd.read_parquet(out / "cards.parquet").shape == cards.shape
assert pd.read_parquet(out / "moves.parquet").shape == moves.shape